# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install "transformers<5" "sentence-transformers<6" "datasets<5" "huggingface-hub<1.0" faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is 

In [ ]:
import warnings

warnings.filterwarnings("ignore", message=".*langchain-community.*")

from typing import List

import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_classic.chains import RetrievalQA

## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

print("Columns:", ds.column_names)
print("Rows:", ds.num_rows)
example = ds[0]
print(f"\nExample source: {example[source_column]}")
print(f"Example text (first 300 chars):\n{example[text_column][:300]}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Columns: ['text', 'source']
Rows: 200

Example source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
Example text (first 300 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-f


In [ ]:
documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column], "doc_id": i},
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

Documents: 200
Example: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx', 'doc_id': 0}
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 



In [ ]:
doc_lengths = pd.Series([len(d.page_content) for d in documents], name="characters")
print(doc_lengths.describe().round(0))
print(f"\nEmpty documents: {(doc_lengths == 0).sum()}")

count       200.0
mean      10226.0
std       28408.0
min          81.0
25%        2326.0
50%        4915.0
75%        9995.0
max      371057.0
Name: characters, dtype: float64

Empty documents: 0


## 2) Split into chunks


In [ ]:
CHUNK_SETTINGS = {"small": (500, 50), "large": (1000, 150)}

def split_documents(chunk_size: int, chunk_overlap: int) -> List[Document]:
    """Split every document into overlapping chunks of at most chunk_size characters."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

splits_by_setting = {
    name: split_documents(*params) for name, params in CHUNK_SETTINGS.items()
}

for name, chunks in splits_by_setting.items():
    print(f"{name:>5}: chunk_size, overlap = {CHUNK_SETTINGS[name]} -> {len(chunks):,} chunks")

chunk_size, chunk_overlap = CHUNK_SETTINGS["small"]
splits = splits_by_setting["small"]
print("\nFirst chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

small: chunk_size, overlap = (500, 50) -> 5,966 chunks
large: chunk_size, overlap = (1000, 150) -> 2,784 chunks

First chunk: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx', 'doc_id': 0}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_WINDOW = 256

emb_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
for name, chunks in splits_by_setting.items():
    n_tokens = np.array([len(emb_tokenizer.encode(c.page_content)) for c in chunks])
    print(
        f"{name:>5}: median {np.median(n_tokens):.0f} tokens, max {n_tokens.max()}, "
        f"over the {EMBEDDING_WINDOW}-token window: {(n_tokens > EMBEDDING_WINDOW).mean():.1%}"
    )

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (630 > 512). Running this sequence through the model will result in indexing errors


small: median 117 tokens, max 501, over the 256-token window: 2.0%
large: median 257 tokens, max 817, over the 256-token window: 50.2%


## 3) Vector store + retriever (FAISS)


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL, encode_kwargs={"normalize_embeddings": True}
)

def build_vectorstore(chunks: List[Document]) -> FAISS:
    """Embed the chunks and store them (with their text and metadata) in FAISS."""
    return FAISS.from_documents(
        chunks, embeddings, distance_strategy=DistanceStrategy.COSINE
    )

vectorstores = {name: build_vectorstore(chunks) for name, chunks in splits_by_setting.items()}
vectorstore = vectorstores["small"]

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever ready


In [ ]:
def show_retrieved(store: FAISS, question: str, k: int = 4, preview: int = 120) -> None:
    """Print the k retrieved chunks with their source file and distance."""
    hits = store.similarity_search_with_score(question, k=k)
    for rank, (doc, distance) in enumerate(hits, start=1):
        source = doc.metadata["source"].split("/")[-1]
        snippet = doc.page_content[:preview].replace("\n", " ")

        print(f"  {rank}. [{distance:.3f}] {source}: {snippet!r}")

In [ ]:
SANITY_QUESTION = "Why does XLA recompile the model when input shapes keep changing?"

for name, store in vectorstores.items():
    print(f"\nChunk setting '{name}' {CHUNK_SETTINGS[name]}:")
    show_retrieved(store, SANITY_QUESTION, k=4)


Chunk setting 'small' (500, 50):
  1. [0.511] perf_train_tpu_tf.md: '#### XLA Rule #3: XLA will need to recompile your model for every different input shape it sees  This is the big one. Wh'
  2. [0.851] perf_train_tpu_tf.md: 'This code has exactly the same effect as the code above, but by avoiding a conditional, we ensure it will compile with X'
  3. [0.883] perf_train_tpu_tf.md: '### How do I make my model XLA compatible?  In many cases, your code is probably XLA-compatible already! However, there '
  4. [0.883] perf_train_tpu_tf.md: 'How can you get around rule #3? The key is **padding** - if you pad all your inputs to the same length, and then use an '

Chunk setting 'large' (1000, 150):
  1. [0.663] perf_train_tpu_tf.md: 'Here, we avoid data-dependent shapes by computing the loss for every position, but zeroing out the masked positions in b'
  2. [1.039] perf_train_tpu_tf.md: 'How can you get around rule #3? The key is **padding** - if you pad all your inputs to the same length, 

In [ ]:
for k in [2, 4, 6]:
    print(f"\nk = {k}:")
    show_retrieved(vectorstore, SANITY_QUESTION, k=k)


k = 2:
  1. [0.511] perf_train_tpu_tf.md: '#### XLA Rule #3: XLA will need to recompile your model for every different input shape it sees  This is the big one. Wh'
  2. [0.851] perf_train_tpu_tf.md: 'This code has exactly the same effect as the code above, but by avoiding a conditional, we ensure it will compile with X'

k = 4:
  1. [0.511] perf_train_tpu_tf.md: '#### XLA Rule #3: XLA will need to recompile your model for every different input shape it sees  This is the big one. Wh'
  2. [0.851] perf_train_tpu_tf.md: 'This code has exactly the same effect as the code above, but by avoiding a conditional, we ensure it will compile with X'
  3. [0.883] perf_train_tpu_tf.md: '### How do I make my model XLA compatible?  In many cases, your code is probably XLA-compatible already! However, there '
  4. [0.883] perf_train_tpu_tf.md: 'How can you get around rule #3? The key is **padding** - if you pad all your inputs to the same length, and then use an '

k = 6:
  1. [0.511] perf_train_tpu_

## 4) Build the RAG chain


In [ ]:
llm_id = "google/flan-t5-small"
hf = pipeline(
    "text2text-generation",
    model=llm_id,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=hf)

def build_qa(k: int = 4, store: FAISS = vectorstore) -> RetrievalQA:
    """RetrievalQA chain that retrieves k chunks and returns them with the answer."""
    return RetrievalQA.from_chain_type(
        llm=llm,
        retriever=store.as_retriever(search_kwargs={"k": k}),
        chain_type="stuff",
        return_source_documents=True,
    )

qa = build_qa(k=4)
print("RAG chain ready")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


RAG chain ready


## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# No-RAG (LLM only)
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# RAG
rag_result = qa.invoke({"query": q})

print("Q:", q)
print("\nNo-RAG answer:\n", no_rag_answer)
print("\nRAG answer:\n", rag_result["result"])
print("\nSources:")
for d in rag_result["source_documents"]:
    print("-", d.metadata.get("source"))

Q: How can I retrieve a model from the Hugging Face Hub?

No-RAG answer:
 Use a hammer to hammer the model into the hammer.

RAG answer:
 Create a model on the [Hugging Face website](https://huggingface.co/join). Create a model on the [Hugging Face website](https://huggingface.co/join). Create a model on the [Hugging Face website](https://huggingface.co/join). Create a model on the [Hugging Face website](https://huggingface.co/docs/hub/how-to-upstream). Create a model on the [Hugging Face

Sources:
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md


In [ ]:
def ask(question: str, k: int = 4) -> None:
    """Answer with RAG and print the sources, to check the answer came from the docs."""
    result = build_qa(k).invoke({"query": question})
    print(f"Q: {question}\nA: {result['result']}\nSources:")
    for doc in result["source_documents"]:
        print(f"  - {doc.metadata['source']}")
    print()

questions = [
    "Why does XLA recompile the model when input shapes keep changing?",
    "How do I choose a metric for evaluation?",
    "What is the Endpoint creation page?",
    "What is the capital city of Uruguay?",
]
for question in questions:
    ask(question)

Q: Why does XLA recompile the model when input shapes keep changing?
A: XLA will need to recompile your model for every different input shapes it sees This is the big one. What this means is that if your input shapes are very variable, XLA will have to recompile your model for every different input shapes it sees This is the big one. What this means is that if your input shapes are very variable, XLA will have to recompile your model for every different input shapes it sees This is the big one. What this means is that if your input shapes are very variable, XLA will have to re
Sources:
  - huggingface/transformers/blob/main/docs/source/en/perf_train_tpu_tf.md
  - huggingface/transformers/blob/main/docs/source/en/perf_train_tpu_tf.md
  - huggingface/transformers/blob/main/docs/source/en/perf_train_tpu_tf.md
  - huggingface/transformers/blob/main/docs/source/en/perf_train_tpu_tf.md

Q: How do I choose a metric for evaluation?
A: **Reading the metric cards** for the relevant metrics and s

Token indices sequence length is longer than the specified maximum sequence length for this model (763 > 512). Running this sequence through the model will result in indexing errors


Q: What is the Endpoint creation page?
A: You can find pricing for Inference Endpoints [here](https://huggingface.co/pricing#endpoints), but a different number is displayed when you deploy a new endpoint from the [GUI](https://ui.endpoints.huggingface.co/new).
Sources:
  - huggingface/blog/blob/main/inference-endpoints-llm.md
  - huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
  - huggingface/blog/blob/main/mantis-case-study.md
  - huggingface/blog/blob/main/mantis-case-study.md

Q: What is the capital city of Uruguay?
A: Uruguay
Sources:
  - gradio-app/gradio/blob/main/demo/question-answering/run.ipynb
  - huggingface/transformers/blob/main/README_ru.md
  - huggingface/transformers/blob/main/README_zh-hans.md
  - huggingface/transformers/blob/main/README_ru.md

